# This is a solution for assignment4

### 1-a-i

This happens when:

The query q is very similar in direction to key $k_j$ (high cosine similarity)
AND/OR key $k_j$ has much larger magnitude than other keys
AND/OR the query q has large magnitude, amplifying the differences in dot products

Essentially, $k_j$ must "match" the query q much better than all other keys do.

### 1-a-ii

Under these conditions, the output c ≈ $v_j$ (approximately equals the value vector $v_j$).

## 1-b 

The query vector should be:
$$q = M \cdot \frac{k_a + k_b}{\|k_a + k_b\|}$$

where M is a large positive scalar.
As M → ∞, the exponential terms dominate, so α_a → ½ and α_b → ½, while α_i → 0 for all other i. (note that $e^0$ = 1)

### 1-c-i

The query vector should be:
$$q = M \cdot \frac{\mu_a + \mu_b}{\|\mu_a + \mu_b\|}$$

where M is a large positive scalar.

Therefore: c ≈ ½v_a + ½v_b = ½(v_a + v_b).

This works because the low variance ensures the key vectors behave predictably like their orthogonal means, allowing us to use the same construction as in the previous part.

### 1-c-ii 

**Analysis of the Modified Covariance**

Let me first understand what's happening with the new covariance structure:

**For key vector k_a:**
- $Σ_a$ = αI + ½($μ_a$ $μ_a^T$)
- This adds extra variance in the direction of $μ_a$
- So $k_a$ ≈ $μ_a$ + noise, where the noise has large variance along $μ_a$ direction

**For other key vectors k_i (i ≠ a):**
- $Σ_i$ = αI (same as before)
- So $k_i$ ≈ $μ_i$ with small spherical noise

Expected Behavior of c Across Different Samples

Using the same query q from part (i):

**The problem:** k_a now has highly variable magnitude while pointing roughly in the same direction as μ_a.

**Dot product analysis:**
- $k_a^T q$ ≈ $||k_a|| · μ_a^T q/||μ_a|| = ||k_a|| · M/||μ_a + μ_b||$
- $k_b^T q$ ≈ $||μ_b|| · M/||μ_a + μ_b|| = M/||μ_a + μ_b||$ (stable)
- Other $k_i^T q$ ≈ 0 (still negligible)

Qualitative Expectations

**High variability in attention weights:**
- When ||k_a|| is large: $α_a >> α_b, so c ≈ v_a$ (mostly copying v_a)
- When ||k_a|| is small: $α_a << α_b, so c ≈ v_b$ (mostly copying v_b)  
- When ||k_a|| ≈ ||μ_a|| = $1: α_a ≈ α_b ≈ ½, so c ≈ ½(v_a + v_b)$ (desired behavior)

**Increased variance in c:**
- Unlike part (i) where c was consistently ≈ $½(v_a + v_b)$
- Now c varies dramatically between samples, ranging from ≈ $v_a$ to ≈ $v_b$
- The output becomes unreliable and unpredictable

**Key insight:** This demonstrates a major drawback of single-headed attention - it's fragile to magnitude variations in key vectors, even when they point in the right direction. Small changes in key norms can completely change which values the attention focuses on, making the model's behavior unstable.

## Appendix

**What is a Covariance Matrix?**

A **covariance matrix** describes how much the components of a random vector vary together. For a d-dimensional random vector, it's a d×d matrix where:

- **Diagonal elements**: Variance of each component
- **Off-diagonal elements**: Covariance between different components

For a random vector **x** = [x₁, x₂, ..., xₐ]ᵀ, the covariance matrix Σ has entries:
- Σᵢⱼ = Cov(xᵢ, xⱼ) for i ≠ j (covariance between components)
- Σᵢᵢ = Var(xᵢ) (variance of component i)

**Understanding Σᵢ = αI**
The formula **Σᵢ = αI** means each covariance matrix is:

$$\Sigma_i = \alpha I = \alpha \begin{bmatrix} 1 & 0 & \cdots & 0 \\ 0 & 1 & \cdots & 0 \\ \vdots & \vdots & \ddots & \vdots \\ 0 & 0 & \cdots & 1 \end{bmatrix} = \begin{bmatrix} \alpha & 0 & \cdots & 0 \\ 0 & \alpha & \cdots & 0 \\ \vdots & \vdots & \ddots & \vdots \\ 0 & 0 & \cdots & \alpha \end{bmatrix}$$

This means:
- **All diagonal elements = α**: Each component has variance α
- **All off-diagonal elements = 0**: Components are uncorrelated
- **Same for all i**: All key vectors have identical covariance structure

**Geometric Interpretation**
When α is "vanishingly small" (α ≈ 0):
- Each key vector kᵢ is distributed very tightly around its mean μᵢ
- The "cloud" of possible values forms a small sphere of radius √α around μᵢ
- Since α ≈ 0, kᵢ ≈ μᵢ with high probability

**Why This Matters**

This setup allows us to treat the random key vectors as if they were deterministic and equal to their means μᵢ, which makes the attention mechanism behave predictably - exactly like the orthogonal case from the previous problem.